In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, f1_score


df = pd.read_csv("loan_approval_data.csv")

# Data Cleaning 

categorical_cols = df.select_dtypes(include=["object"]).columns
numerical_cols = df.select_dtypes(include=["number"]).columns


num_imp = SimpleImputer(strategy="mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])


cat_imp = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])


df = df.drop("Applicant_ID", axis=1)

# Data preprocessing 

le = LabelEncoder()
df["Education_Level"] = le.fit_transform(df["Education_Level"])
df["Loan_Approved"] = le.fit_transform(df["Loan_Approved"])


cols = ["Employment_Status", "Marital_Status", "Loan_Purpose", "Property_Area", "Gender", "Employer_Category"]
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
encoded = ohe.fit_transform(df[cols])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cols), index=df.index)
df = pd.concat([df.drop(columns=cols), encoded_df], axis=1)

# Feature Engineering 

df["DTI_ratio_sq"] = df["DTI_Ratio"]**2
df["Credit_Score_sq"] = df["Credit_Score"]**2


X = df.drop(columns=["DTI_Ratio", "Credit_Score", "Loan_Approved"])
y = df["Loan_Approved"]

# train test split 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

# feature scaling 

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Naive Bayes

nb_model = GaussianNB()
nb_model.fit(X_train_scaled, y_train)

y_pred = nb_model.predict(X_test_scaled)

# evalution 

from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, f1_score

print("Naive Bayes model evalution")
print("accuracy score : ", accuracy_score(y_test, y_pred))
print("precision score : ", precision_score(y_test, y_pred))
print("f1 score : ", f1_score(y_test, y_pred))
print("recall score : ", recall_score(y_test, y_pred))
print("CM : ", confusion_matrix(y_test, y_pred))


Naive Bayes model evalution
accuracy score :  0.865
precision score :  0.7833333333333333
f1 score :  0.7768595041322314
recall score :  0.7704918032786885
CM :  [[126  13]
 [ 14  47]]
